일상용어

In [ ]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

base_url = "https://sldict.korean.go.kr/front/sign/signList.do"

headers = {
    "User-Agent": "Mozilla/5.0"
}

session = requests.Session()
session.headers.update(headers)

# 네가 확인한 카테고리 코드
category_map = {
    "인간": "CTE001",
    "삶": "CTE002",
    "식생활": "CTE003",
    "의생활": "CTE004",
    "주생활": "CTE005",
    "사회생활": "CTE006",
    "경제생활": "CTE007",
    "교육": "CTE008",
    "나라명및지명": "CTE009",
    "종교": "CTE010",
    "문화": "CTE011",
    "정치와행정": "CTE012",
    "자연": "CTE013",
    "동식물": "CTE014",
    "개념": "CTE015",
    "기타": "CTE016",
}

all_results = []
seen = set()

for category_name, category_code in category_map.items():
    print(f"\n[{category_name}] 수집 시작")
    
    page = 1
    while True:
        params = {
            "current_pos_index": "",
            "origin_no": "0",
            "searchWay": "",
            "top_category": "",
            "category": category_code,
            "detailCategory": "",
            "searchKeyword": "",
            "pageIndex": page,
            "pageJumpIndex": ""
        }

        response = session.get(base_url, params=params, timeout=20)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        items = soup.select("div.li_w span.tit a")

        if not items:
            print(f"  {page}페이지: 결과 없음, 종료")
            break

        page_count = 0

        for a_tag in items:
            word = a_tag.get_text(strip=True)
            href = a_tag.get("href", "")

            match = re.search(r"fnSearchContentsView\('(\d+)',\s*'([^']*)'\)", href)
            if match:
                origin_no = match.group(1)
                inner_category = match.group(2)

                key = (word, origin_no, inner_category)
                if key not in seen:
                    seen.add(key)
                    all_results.append({
                        "category_name": category_name,
                        "category_code": category_code,
                        "word": word,
                        "origin_no": origin_no,
                        "detail_category": inner_category,
                        "pageIndex": page
                    })
                    page_count += 1

        print(f"  {page}페이지 수집: {page_count}개")

        # 너무 빠르게 요청하지 않도록 약간 대기
        time.sleep(0.5)
        page += 1

df = pd.DataFrame(all_results)
df.to_csv("ksl_gloss_dictionary_all.csv", index=False, encoding="utf-8-sig")

print(f"\n총 {len(df)}개 저장 완료")
print(df.head())


[인간] 수집 시작
  1페이지 수집: 10개
  2페이지 수집: 10개
  3페이지 수집: 10개
  4페이지 수집: 10개
  5페이지 수집: 10개
  6페이지 수집: 10개
  7페이지 수집: 10개
  8페이지 수집: 10개
  9페이지 수집: 10개
  10페이지 수집: 10개
  11페이지 수집: 10개
  12페이지 수집: 10개
  13페이지 수집: 10개
  14페이지 수집: 10개
  15페이지 수집: 10개
  16페이지 수집: 10개
  17페이지 수집: 10개
  18페이지 수집: 10개
  19페이지 수집: 10개
  20페이지 수집: 10개
  21페이지 수집: 10개
  22페이지 수집: 10개
  23페이지 수집: 10개
  24페이지 수집: 10개
  25페이지 수집: 10개
  26페이지 수집: 10개
  27페이지 수집: 10개
  28페이지 수집: 10개
  29페이지 수집: 10개
  30페이지 수집: 10개
  31페이지 수집: 10개
  32페이지 수집: 2개
  33페이지: 결과 없음, 종료

[삶] 수집 시작
  1페이지 수집: 10개
  2페이지 수집: 10개
  3페이지 수집: 10개
  4페이지 수집: 10개
  5페이지 수집: 10개
  6페이지 수집: 10개
  7페이지 수집: 10개
  8페이지 수집: 10개
  9페이지 수집: 10개
  10페이지 수집: 10개
  11페이지 수집: 10개
  12페이지 수집: 10개
  13페이지 수집: 10개
  14페이지 수집: 10개
  15페이지 수집: 10개
  16페이지 수집: 10개
  17페이지 수집: 10개
  18페이지 수집: 10개
  19페이지 수집: 10개
  20페이지 수집: 10개
  21페이지 수집: 10개
  22페이지 수집: 10개
  23페이지 수집: 10개
  24페이지 수집: 10개
  25페이지 수집: 8개
  26페이지: 결과 없음, 종료

[식생활] 수집 시작
  1페이지 수집: 10개
  2페이지 수집: 10개
  3페

In [3]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

df = pd.read_csv("ksl_gloss_dictionary_all.csv")

df.head(5)

,category_name,category_code,word,origin_no,detail_category,pageIndex
0,인간,CTE001,"못생기다,못나다,추하다,밉다",12482,0,1
1,인간,CTE001,"오해,곡해",8146,1,1
2,인간,CTE001,"배부르다,부르다",11382,2,1
3,인간,CTE001,어지럽다,8219,3,1
4,인간,CTE001,"넣다,담그다",6631,4,1


In [ ]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

base_detail_url = "https://sldict.korean.go.kr/front/sign/signContentsView.do"

headers = {
    "User-Agent": "Mozilla/5.0"
}

session = requests.Session()
session.headers.update(headers)

df = pd.read_csv("ksl_gloss_dictionary_all.csv")

combination_info_list = []
korean_equivalent_list = []
description_list = []

for i, row in df.iterrows():
    origin_no = str(row["origin_no"])
    category_code = str(row["category_code"])
    page_index = str(row["pageIndex"])

    params = {
        "current_pos_index": "0",
        "origin_no": origin_no,
        "searchWay": "",
        "top_category": "",
        "category": category_code,
        "detailCategory": "",
        "searchKeyword": "",
        "pageIndex": page_index,
        "pageJumpIndex": ""
    }

    try:
        response = session.get(base_detail_url, params=params, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        lines = [line.strip() for line in soup.get_text("\n", strip=True).split("\n") if line.strip()]

        combination_info = None
        korean_equivalent = None
        description = None

        for idx, line in enumerate(lines):
            if line == "결합정보" and idx + 1 < len(lines):
                combination_info = lines[idx + 1]
            elif line == "한국어 대응표현" and idx + 1 < len(lines):
                korean_equivalent = lines[idx + 1]
            elif line == "수형 설명" and idx + 1 < len(lines):
                description = lines[idx + 1]

        combination_info_list.append(combination_info)
        korean_equivalent_list.append(korean_equivalent)
        description_list.append(description)

        print(f"{i+1}/{len(df)} 완료 - origin_no={origin_no} / 결합정보={combination_info}")

    except Exception as e:
        print(f"{i+1}/{len(df)} 실패 - origin_no={origin_no} / {e}")
        combination_info_list.append(None)
        korean_equivalent_list.append(None)
        description_list.append(None)

    time.sleep(0.5)

df["combination_info"] = combination_info_list
df["korean_equivalent"] = korean_equivalent_list
df["hand_description"] = description_list

df.to_csv("ksl_gloss_dictionary_with_detail.csv", index=False, encoding="utf-8-sig")
print("저장 완료")

1/3507 완료 - origin_no=12482 / 결합정보=None
2/3507 완료 - origin_no=8146 / 결합정보=None
3/3507 완료 - origin_no=11382 / 결합정보=[배/두껍다]
4/3507 완료 - origin_no=8219 / 결합정보=None
5/3507 완료 - origin_no=6631 / 결합정보=None
6/3507 완료 - origin_no=2673 / 결합정보=None
7/3507 완료 - origin_no=10741 / 결합정보=None
8/3507 완료 - origin_no=10377 / 결합정보=None
9/3507 완료 - origin_no=7859 / 결합정보=None
10/3507 완료 - origin_no=7664 / 결합정보=[위대하다+사람]
11/3507 완료 - origin_no=7469 / 결합정보=None
12/3507 완료 - origin_no=7327 / 결합정보=None
13/3507 완료 - origin_no=7240 / 결합정보=[울다+괴롭다]
14/3507 완료 - origin_no=6016 / 결합정보=[성냥+불]
15/3507 완료 - origin_no=5224 / 결합정보=None
16/3507 완료 - origin_no=2039 / 결합정보=None
17/3507 완료 - origin_no=1818 / 결합정보=None
18/3507 완료 - origin_no=528 / 결합정보=[특별+성격]
19/3507 완료 - origin_no=15526 / 결합정보=[사용+사람]
20/3507 완료 - origin_no=14866 / 결합정보=None
21/3507 완료 - origin_no=12013 / 결합정보=None
22/3507 완료 - origin_no=11778 / 결합정보=None
23/3507 완료 - origin_no=11619 / 결합정보=None
24/3507 완료 - origin_no=11518 / 결합정보=None
25/3507 완료 - origin_

In [ ]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

df = pd.read_csv("ksl_gloss_dictionary_with_detail.csv")

df.head(15)

,category_name,category_code,word,origin_no,detail_category,pageIndex,combination_info,korean_equivalent,hand_description
0,인간,CTE001,"못생기다,못나다,추하다,밉다",12482,0,1,NaN,"못생기다,못나다,추하다,밉다",얼굴 앞에서 오므렸던 오른손을 얼굴 쪽으로 끌어들이며 편다.
1,인간,CTE001,"오해,곡해",8146,1,1,NaN,"오해,곡해",오므린 두 손을 손등이 밖으로 향하게 하여 얼굴 앞에 나란히 세웠다가 두 팔이 ‘X...
2,인간,CTE001,"배부르다,부르다",11382,2,1,[배/두껍다],"배부르다,부르다",두 손 각각 손가락을 모아 붙여 끝이 옆으로 향하게 하여 5지 등을 배에 대고 1·...
3,인간,CTE001,어지럽다,8219,3,1,NaN,어지럽다,오른손의 손가락을 모아 끝이 안으로 향하게 하여 눈앞에서 왼쪽으로 두 바퀴 돌린다.
4,인간,CTE001,"넣다,담그다",6631,4,1,NaN,"넣다,담그다",손끝이 오른쪽으로 향하게 세운 왼손의 1·5지 사이에 손끝이 아래로 향하게 편 오른...
5,인간,CTE001,"뽀뽀,입맞춤,키스,맞추다",2673,5,1,NaN,"뽀뽀,입맞춤,키스,맞추다","손가락 끝을 모아 쥔 오른손의 손끝을 입술에 댄 다음, 손가락 등이 위로 향하게 손..."
6,인간,CTE001,"솔직하다,정직",10741,6,1,NaN,"솔직하다,정직","1·5지 끝을 맞대어 동그라미를 만든 두 주먹을, 오른 주먹이 위에 놓이게 배에 대..."
7,인간,CTE001,"뜨다,개안",10377,7,1,NaN,"뜨다,개안",오른 주먹의 1·5지를 펴서 붙여 오른쪽 눈앞에서 뗀다.
8,인간,CTE001,울다,7859,8,1,NaN,울다,양 주먹의 1·5지 끝을 맞대어 양 눈 밑에서 아래로 두 번 내린다.
9,인간,CTE001,위인,7664,9,1,[위대하다+사람],위인,오른 주먹의 1·5지를 펴서 끝을 맞대어 머리 오른쪽에 댔다가 1지를 위로 편 다음...


사전 정리

In [7]:
import pandas as pd

# 파일 불러오기
df_daily = pd.read_csv("ksl_gloss_dictionary_with_detail.csv")

# 필요한 열만 선택 + 이름 변경
df_daily_merged = df_daily[["word", "combination_info"]].copy()
df_daily_merged = df_daily_merged.rename(columns={
    "word": "word",
    "combination_info": "combination"
})

# 열 순서 정리
df_merged = df_daily_merged[["word", "combination"]]
print("샘플 수 확인:", len(df_merged))

# 완전히 동일한 행 중복 제거
df_merged = df_merged.drop_duplicates().reset_index(drop=True)
print("동일 행 제거 후 샘플 수:", len(df_merged))

#word가 비어있는 행 제거
df_merged = df_merged.dropna(subset=["word"]).reset_index(drop=True)
print("한국 단어 비어 있는 행 제거 후 샘플 수", len(df_merged))


# 저장
df_merged.to_csv("ksl_gloss_dictionary.csv", index=False, encoding="utf-8-sig")
print(df_merged.head())

샘플 수 확인: 3507
동일 행 제거 후 샘플 수: 3479
한국 단어 비어 있는 행 제거 후 샘플 수 3479
              word combination
0  못생기다,못나다,추하다,밉다         NaN
1            오해,곡해         NaN
2         배부르다,부르다     [배/두껍다]
3             어지럽다         NaN
4           넣다,담그다         NaN


word 여러 개 있는 거 분리

In [10]:
import pandas as pd

df = pd.read_csv("ksl_gloss_dictionary.csv")

expanded_rows = []

for _, row in df.iterrows():
    words = str(row["word"]).split(",")

    for w in words:
        w = w.strip()
        if w and w != "nan":
            expanded_rows.append({
                "word": w,
                "combination": row["combination"]
            })

df_split = pd.DataFrame(expanded_rows)

# 완전히 동일한 것만 제거
df_split = df_split.drop_duplicates().reset_index(drop=True)

df_split.to_csv("ksl_gloss_dictionary_split.csv", index=False, encoding="utf-8-sig")

print(df_split.head())
print("총 단어 수:", len(df_split))

   word combination
0  못생기다         NaN
1   못나다         NaN
2   추하다         NaN
3    밉다         NaN
4    오해         NaN
총 단어 수: 6625


매칭 쉽게 워드 컬럼 정리

In [12]:
import re
import pandas as pd

df = pd.read_csv("ksl_gloss_dictionary_split.csv")

def normalize_for_mapping(text):
    if pd.isna(text):
        return pd.Series([None, None])

    raw_word = str(text).strip()

    if raw_word == "" or raw_word.lower() == "nan":
        return pd.Series([raw_word, None])

    word = raw_word

    # ---------------------------------------------------
    # 1. 맨 앞 괄호 제거: (시설물)다리 → 다리
    # ---------------------------------------------------
    word = re.sub(r'^\([가-힣\s]+\)\s*', '', word)

    # ---------------------------------------------------
    # 2. 맨 뒤 괄호 제거: 망신(당하다) → 망신
    # ---------------------------------------------------
    word = re.sub(r'\([가-힣\s]+\)\s*$', '', word)

    # ---------------------------------------------------
    # 3. 중간 괄호는 내용만 살림: 면목(이) 없다 → 면목이 없다
    # ---------------------------------------------------
    word = re.sub(r'\(([^)]+)\)', r'\1', word)

    # ---------------------------------------------------
    # 4. 한국어 + 공백만 남김
    # ---------------------------------------------------
    word = re.sub(r'[^가-힣\s]', '', word)

    # ---------------------------------------------------
    # 5. 공백 정리
    # ---------------------------------------------------
    word = re.sub(r'\s+', ' ', word).strip()

    if word == "":
        word = None

    return pd.Series([raw_word, word])

df[["raw_word", "word"]] = df["word"].apply(normalize_for_mapping)

# 필요한 열만
keep_cols = []
for col in ["domain_type", "raw_word", "word", "combination"]:
    if col in df.columns:
        keep_cols.append(col)

df_final = df[keep_cols].copy()

# 빈 값 제거
df_final = df_final[df_final["word"].notna()]
df_final = df_final[df_final["word"].astype(str).str.strip() != ""]

# 중복 제거
df_final = df_final.drop_duplicates().reset_index(drop=True)

df_final.to_csv("ksl_gloss_dictionary_for_mapping.csv", index=False, encoding="utf-8-sig")

print(df_final.head(20))
print("총 행 수:", len(df_final))

   raw_word  word combination
0      못생기다  못생기다         NaN
1       못나다   못나다         NaN
2       추하다   추하다         NaN
3        밉다    밉다         NaN
4        오해    오해         NaN
5        곡해    곡해         NaN
6      배부르다  배부르다     [배/두껍다]
7       부르다   부르다     [배/두껍다]
8      어지럽다  어지럽다         NaN
9        넣다    넣다         NaN
10      담그다   담그다         NaN
11       뽀뽀    뽀뽀         NaN
12      입맞춤   입맞춤         NaN
13       키스    키스         NaN
14      맞추다   맞추다         NaN
15     솔직하다  솔직하다         NaN
16       정직    정직         NaN
17       뜨다    뜨다         NaN
18       개안    개안         NaN
19       울다    울다         NaN
총 행 수: 6585
